# Explore the LNMC Layer 5 Pyramidal Connectivity Whole-Cell Patch-Clamp Dataset (Comprehensive)

## Introduction
This notebook provides a comprehensive exploratory analysis of the **Multineuron Whole-Cell Patch-Clamp Dataset of Layer 5 Pyramidal Synaptic Connectivity in Juvenile Rat Somatosensory Cortex**, produced at the EPFL Laboratory of Neural Microcircuitry (LNMC) and described in Perin, Berger & Markram (2011, PNAS).

Learn more:
- Data Package DOI: [10.71728/senscience.t3x9-1z2t](https://sen.science/doi/10.71728/senscience.t3x9-1z2t/)
- Source article: [10.1073/pnas.1016051108](https://doi.org/10.1073/pnas.1016051108)

As a FAIR² Data Package, it ensures accessibility, interoperability, and AI-readiness. FAIR² datasets follow the MLCommons **Croissant** 🥐 format for machine learning datasets. See the [MLCommons Croissant Format Specification](https://docs.mlcommons.org/croissant/docs/croissant-spec.html).

The dataset provides simultaneous whole-cell patch-clamp electrophysiology from up to **twelve layer 5 pyramidal neurons** recorded at once (a purpose-built “12-patch” setup):
- **47 recording sessions**, one per multineuron experiment
- **150,121 current-clamp acquisition sweeps** across 160 distinct Igor-Pro stimulus protocols
- **482 patched-cell electrode configurations**

Three tidy Parquet record-sets index the collection: `session_metadata` (one row per session), `acquisition_series` (one row per sweep), and `electrode_metadata` (one row per patched cell). This notebook loads them via the Croissant metadata, links every column back to its method step, and renders a multineuron current-clamp viewer plus a spike-triggered-average connectivity readout — reading the raw voltage waveforms on demand from the canonical **NWB** files.

### Install and import required libraries

In [ ]:
# Install mlcroissant and the loading/plotting/NWB dependencies
!pip install mlcroissant pandas pyarrow matplotlib seaborn h5py fsspec aiohttp

In [ ]:
import json, re, zipfile, tempfile, shutil, urllib.request
from pathlib import Path
from collections import defaultdict

import fsspec
import mlcroissant as mlc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
import matplotlib.pyplot as plt

try:
    import h5py
except ImportError:
    h5py = None

from IPython.display import Markdown, display

sns.set_theme(style="whitegrid")

## 1. Data Loading
We use the `mlcroissant` library to read the FAIR² `fair2.json` metadata, then load the three tidy Parquet record-sets it declares into `pandas` DataFrames. All three are index tables and fit comfortably in memory (the largest, `acquisition_series`, has 150,121 rows). The **raw voltage waveforms** are not in these tables — they remain in the 48 canonical NWB files and are read on demand in the electrophysiology viewer below. Every column is linked back to a specific method step via the `method_description_object` entries in `schema.json`.

In [ ]:
# Load the dataset from the Croissant metadata file (mlcroissant), with a robust
# fallback to reading the Parquet index tables directly if the Croissant runtime
# cannot parse the metadata in this environment.
# The published package: its fair2.json, record sets and resources
BASE = 'https://sen.science/doi/10.71728/senscience.t3x9-1z2t'
try:
    ds = mlc.Dataset(f'{BASE}/fair2.json')
except Exception as e:
    print(f'mlcroissant could not parse fair2.json here ({type(e).__name__}); '
          f'falling back to direct Parquet loading.')
    ds = None

def resolve_data_dir():
    candidates = [
        'data/parquet',
        'resources/data/parquet',
        '../wrangled/perin_et_al_2011_patch_clamp_connectivity_nwb_dataset_blue_brain_project/data/parquet',
    ]
    for c in candidates:
        if Path(c).is_dir():
            return c
    hits = list(Path('.').rglob('acquisition_series.parquet'))
    return str(hits[0].parent) if hits else 'data/parquet'

DATA_DIR = f'{BASE}/resources/record-sets'
print(f'Parquet directory: {DATA_DIR}')

# Record-set names come from the Croissant metadata when available, else the raw JSON.
if ds is not None:
    rs_by_name = {rs.name.replace('.parquet', ''): rs for rs in ds.metadata.record_sets}
    rs_names = list(rs_by_name)
else:
    fair2 = json.loads(urllib.request.urlopen(f'{BASE}/fair2.json').read())
    rs_names = [rs['name'].replace('.parquet', '') for rs in fair2.get('recordSet', [])]
    rs_by_name = {}
# ensure every Parquet present is loaded even if not declared
for p in sorted(Path(DATA_DIR).glob('*.parquet')):
    if p.stem not in rs_names:
        rs_names.append(p.stem)

dataframes = {}
for name in rs_names:
    df = None
    if name in rs_by_name:
        try:
            df = pd.DataFrame(list(ds.records(rs_by_name[name].id)))
            df.columns = [col.split('/')[-1] for col in df.columns]
        except Exception:
            df = None
    if df is None:
        df = pd.read_parquet(f'{DATA_DIR}/{name}.parquet')
    dataframes[name] = df

session_metadata   = dataframes['session_metadata']
acquisition_series = dataframes['acquisition_series']
electrode_metadata = dataframes['electrode_metadata']
stimulus_series    = dataframes.get('stimulus_series')   # injected current (recorded + reconstructed)

# mlcroissant sometimes loads text columns as bytes — normalise here
for df in [d for d in dataframes.values() if d is not None]:
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

print("\nRecord sets loaded (mlcroissant / Parquet):")
for name, df in dataframes.items():
    print(f"  {name:<18}: {len(df):>8,} rows x {len(df.columns)} cols")


### 1.1 Data dictionary with method-step provenance
Every column in every table is linked to a specific method step via the `schema.json`'s `method_description_object`. Here we surface the linkage in tabular form.

In [ ]:
# Load the schema.json to expose the data dictionary + method step links
schema = json.loads(urllib.request.urlopen(f'{BASE}/schema.json').read())

# Method-step / section catalogue
section_index = {s['id']: s.get('name', '') for s in schema.get('methods_description', [])}
print(f"{len(section_index)} method sections declared in schema.methods_description")

# Build a wide table: (table, column, dataType, unit, → method step)
rows = []
for row in schema['data_dictionary']['data_dictionary_rows']:
    tname = row['file_object']['name'].replace('.parquet', '')
    for f in row['fields']:
        md_obj = f.get('method_description_object', {}) or {}
        rows.append({
            'table':       tname,
            'column':      f['variable_name'],
            'dataType':    f.get('dataType', '').split('/')[-1],
            'unit':        f.get('unit', {}).get('symbol', ''),
            'method_step': md_obj.get('name', '—'),
        })
dd_df = pd.DataFrame(rows)
display(Markdown(f'### {len(dd_df)} variables across {dd_df.table.nunique()} tables — each linked to a method section'))
display(dd_df)

### 1.2 Dataset Metadata
Inspect the dataset's top-level metadata to confirm name, DOI, and licence.

In [ ]:
if ds is not None:
    metadata = ds.metadata.to_json()
    creators = [c.name for c in ds.metadata.creators]
else:
    metadata = json.loads(urllib.request.urlopen(f'{BASE}/fair2.json').read())
    creators = []
    for c in (metadata.get('creator') or metadata.get('schema:creator') or []):
        creators.append(c.get('name', str(c)) if isinstance(c, dict) else str(c))

name        = metadata.get('name', 'N/A')
description = metadata.get('description', 'N/A')
if isinstance(description, list):
    description = description[0].get('@value', description[0]) if description else 'N/A'

print(f"Name       : {name}")
print(f"Version    : {metadata.get('version', 'N/A')}")
print(f"License    : {metadata.get('license', 'N/A')}")
print(f"URL        : {metadata.get('url', 'N/A')}")
print(f"Creators   : {len(creators)}")
for c in creators:
    print(f"    - {c}")
print(f"\nDescription: {description}")


## 2. Data Preparation and Integration
We assemble a per-session integrated summary by aggregating the sweep-level table (`acquisition_series`) and the electrode-level table (`electrode_metadata`) onto the session-level table (`session_metadata`), keyed on the canonical `session_id`.

In [ ]:
# --- Per-session sweep summary --- #
acquisition_series['channel_num'] = pd.to_numeric(acquisition_series['channel_num'], errors='coerce')
sweep_summary = (
    acquisition_series.groupby('session_id')
        .agg(n_sweeps=('acquisition_name', 'size'),
             n_protocols=('protocol', 'nunique'),
             n_channels=('channel_num', lambda s: s[s >= 0].nunique()))
        .reset_index()
)

# --- Per-session electrode count --- #
elec_summary = (electrode_metadata.groupby('session_id')
                .agg(n_cells=('cell_index', 'nunique')).reset_index())

# Merge onto session-level metadata
recording_summary = (session_metadata
                     .merge(sweep_summary, on='session_id', how='left')
                     .merge(elec_summary, on='session_id', how='left'))

# Parse experiment-type tag and experimental date from the session_id
def paradigm(sid):
    m = re.match(r'^\d{6}([A-Za-z]+\d*)', sid)
    t = m.group(1) if m else 'other'
    return 'Plast' if t in ('Plast', 'Plastic') else t
recording_summary['paradigm'] = recording_summary['session_id'].map(paradigm)

display(Markdown('### Recording summary (per session)'))
display(recording_summary[['session_id', 'paradigm', 'n_electrodes', 'n_cells',
                           'n_acquisitions', 'n_sweeps', 'n_protocols', 'n_channels']].head())

## 3. Exploratory Data Analysis (EDA)

### 3.1 Overview of the Integrated Dataset

In [ ]:
print("Recording summary:")
print(f"  {recording_summary.shape[0]} sessions × {recording_summary.shape[1]} fields")
print(f"  paradigm distribution : {dict(recording_summary['paradigm'].value_counts())}")
print(f"  cells per session     : {int(recording_summary['n_electrodes'].min())}–{int(recording_summary['n_electrodes'].max())} "
      f"(median {recording_summary['n_electrodes'].median():.0f})")
print()
print("Sweep-level:")
print(f"  {len(acquisition_series):,} acquisition sweeps across {acquisition_series['session_id'].nunique()} sessions")
print(f"  distinct stimulus protocols : {acquisition_series['protocol'].nunique():,}")
print(f"  distinct sweep identifiers  : {acquisition_series['sweep_id'].nunique():,}")
print()
print("Electrode-level:")
print(f"  {len(electrode_metadata):,} patched-cell electrode configurations across {electrode_metadata['session_id'].nunique()} sessions")

### 3.2 Data Visualization

#### 3.2.1 Recording composition
How the sessions are composed: the number of cells patched simultaneously per session, and the number of acquisition sweeps per session.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

vc = recording_summary['n_electrodes'].value_counts().sort_index()
axes[0].bar(vc.index.astype(int).astype(str), vc.values, color='#4C72B0', edgecolor='white')
axes[0].set_xlabel('Cells patched per session'); axes[0].set_ylabel('# sessions')
axes[0].set_title(f'Simultaneously recorded cells (median {int(recording_summary["n_electrodes"].median())})')

axes[1].hist(recording_summary['n_acquisitions'].dropna(), bins=12, color='#DD8452', edgecolor='white')
axes[1].axvline(recording_summary['n_acquisitions'].median(), color='#4C72B0', ls='--',
                label=f"median {int(recording_summary['n_acquisitions'].median()):,}")
axes[1].set_xlabel('Sweeps per session'); axes[1].set_ylabel('# sessions')
axes[1].set_title('Acquisition volume per session'); axes[1].legend()

plt.tight_layout(); plt.show()

#### 3.2.2 The stimulus repertoire
Sweeps are distributed across 160 distinct Igor-Pro stimulus protocols. Here we show the most-represented protocols by total sweep count.

In [ ]:
proto = (acquisition_series.groupby('protocol')
            .agg(n_sweeps=('acquisition_name', 'size'),
                 n_sessions=('session_id', 'nunique'))
            .sort_values('n_sweeps', ascending=False))

top = proto.head(20)[::-1]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top.index, top['n_sweeps'], color='#8172B3', edgecolor='white')
for y, (n, r) in enumerate(zip(top['n_sweeps'], top['n_sessions'])):
    ax.text(n + top['n_sweeps'].max()*0.01, y, f'{n:,} ({r} sess)', va='center', fontsize=7)
ax.set_xlabel('Sweeps'); ax.set_xlim(0, top['n_sweeps'].max() * 1.2)
ax.set_title(f'Stimulus repertoire — 20 most-represented protocols (of {acquisition_series["protocol"].nunique():,})')
plt.tight_layout(); plt.show()

#### 3.2.3 Electrophysiology viewer — injected current & mean response per cell
The curated NWB files carry the experimenters' own **pre-computed trial-averaged responses** (`igor_source_experiment/Packed_Data/Stimulated_{P}G{R}Avg`): one presynaptic cell is driven with a current-pulse train while the mean membrane-potential response of every simultaneously patched cell is recorded. We read these averages directly (no sweep-by-sweep playback and no re-computation) and show (A) the **current injected** into the driven cell and (B) the **mean response of each cell**. A genuine monosynaptic connection shows up as a sustained postsynaptic depolarisation.

In [ ]:
# --- Electrophysiology viewer: injected current + mean response per cell --- #
# The curated NWB files carry, per session, the experimenters' pre-computed
# trial-averaged responses:  igor_source_experiment/Packed_Data/Stimulated_{P}G{R}Avg
#   → stimulate presynaptic cell (P-1), record the trial-averaged Vm in cell R.
# We show (A) the current injected into the driven cell and (B) the mean
# response of every simultaneously patched cell — no sweep-by-sweep playback.

EXAMPLE_SESSION = '070521L5_1'          # session with clean averages (data-article Fig. 4)
FS = 5000.0                             # sampling rate (Hz)
PRE_CELL = 5                            # presynaptic (driven) cell to feature

AVG_RE  = re.compile(r'^Stimulated_(\d+)G(\d+)Avg$')
NAME_RE = re.compile(r'^(?P<sw>[A-Za-z]*\d+)_(?P<pr>.+)_ch(?P<ch>\d+)_(?P<rep>\d+)$')

def resolve_nwb(fname):
    """Find an NWB file locally, or extract it from the package zip on demand."""
    for d in ['resources/artifacts', 'nwb', 'resources/nwb', 'assets/nwb',
              '../wrangled/perin_et_al_2011_patch_clamp_connectivity_nwb_dataset_blue_brain_project/assets/nwb']:
        p = Path(d) / fname
        if p.exists():
            return str(p)
    # The published package zip, read remotely: only its index and this file are downloaded
    zpath = f'{BASE}/resources/artifacts/nwb_ephys_files.zip'
    with fsspec.open(zpath) as remote:
        with zipfile.ZipFile(remote) as z:
            if fname in z.namelist():
                out = Path(tempfile.gettempdir()) / fname
                if not out.exists():
                    print(f'Extracting {fname} from package zip (one-off)...')
                    with z.open(fname) as src, open(out, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
                return str(out)
    return None

NWB_PATH = resolve_nwb(EXAMPLE_SESSION + '.nwb') if h5py else None
if NWB_PATH is None:
    print('NWB file not available in this runtime — skipping the trace viewer.')
    print('Provide the NWB files (or the package zip) to run this cell.')
else:
    print(f'Reading curated averages + injected current from: {NWB_PATH}')
    with h5py.File(NWB_PATH, 'r') as h:
        # ---- (1) pre-computed trial-averaged responses (volts) ----
        pdp = h.get('igor_source_experiment/Packed_Data')
        by_pipette = defaultdict(dict)
        if pdp is not None:
            for k in pdp:
                m = AVG_RE.match(k)
                if m:
                    by_pipette[int(m.group(1))][int(m.group(2))] = pdp[k][:]

        if PRE_CELL + 1 not in by_pipette:
            avail = sorted(p - 1 for p in by_pipette)
            PRE_CELL = avail[0] if avail else PRE_CELL
        resp = by_pipette.get(PRE_CELL + 1, {})

        # keep only physiologically patched cells (rest between -85 and -50 mV)
        def rest_mV(v): return np.median(v[:200]) * 1000.0
        patched = sorted(R for R, v in resp.items() if -85 < rest_mV(v) < -50)
        driven  = [R for R in patched if resp[R].max() > 0]      # cell(s) that spiked
        driven  = driven[0] if driven else PRE_CELL

        # ---- (2) injected current into the driven cell (drive sweeps only) ----
        stim = h.get('stimulus/presentation')
        cur_ch = 2 * PRE_CELL + 1
        inj = None
        if stim is not None:
            names = [n for n in stim if (m := NAME_RE.match(n)) and int(m.group('ch')) == cur_ch]
            if names:
                amps = np.array([np.ptp(stim[n]['data'][:]) for n in names])
                sel  = [names[i] for i in np.where(amps >= amps.max() * 0.5)[0]]  # actually-driven sweeps
                L    = min(stim[n]['data'].shape[0] for n in sel)
                inj  = np.vstack([stim[n]['data'][:L] for n in sel]).mean(0) * 1e9  # nA

    n  = len(resp[patched[0]]); t = np.arange(n) / FS * 1000.0                      # ms
    fig, (axA, axB) = plt.subplots(2, 1, figsize=(11, 8),
                                   gridspec_kw={'height_ratios': [1, 3.2]}, sharex=True)

    # ---- Panel A: injected current ----
    if inj is not None:
        ti = np.arange(len(inj)) / FS * 1000.0
        axA.plot(ti, inj, color='#b5322e', lw=1.0)
        axA.set_ylabel('injected\ncurrent (nA)')
        axA.set_title(f'A  Current injected into the driven cell {driven}  '
                      f'(mean of {len(sel)} drive sweeps, 8-pulse train)', loc='left', fontsize=10)
    else:
        axA.text(0.5, 0.5, 'no injected-current channel recorded for this cell',
                 ha='center', va='center', transform=axA.transAxes, color='#888')
    for s in ['top', 'right']: axA.spines[s].set_visible(False)

    # ---- Panel B: mean response of every patched cell (stacked) ----
    off = 12.0
    for i, R in enumerate(patched):
        v = (resp[R] - np.median(resp[R][:200])) * 1000.0        # baseline-subtracted mV
        y = i * off
        if R == driven:
            col, lab = '#b5322e', f'cell {R} — driven (mean spike train)'
            v = np.clip(v, -off * 0.4, off * 0.9)                # clip spikes to keep layout
        else:
            col, lab = '#2c5f8a', f'cell {R}'
        axB.plot(t, v + y, lw=0.8, color=col)
        axB.text(-18, y, f'cell {R}', ha='right', va='center', fontsize=8,
                 color=col, fontweight='bold' if R == driven else 'normal')
    axB.set_yticks([]); axB.set_xlabel('time (ms)')
    axB.set_xlim(-40, t.max())
    axB.set_title(f'B  Mean response per simultaneously patched cell '
                  f'({len(patched)} cells) — pre-computed trial averages', loc='left', fontsize=10)
    for s in ['top', 'right', 'left']: axB.spines[s].set_visible(False)
    fig.suptitle(f'Injected current & mean synaptic response · {EXAMPLE_SESSION} · '
                 f'stimulate cell {driven}', y=0.99, fontsize=12)
    plt.tight_layout(); plt.show()

    display(Markdown(
        f'*Curated trial-averaged responses read directly from the NWB file '
        f'(`igor_source_experiment/Packed_Data/Stimulated_*Avg`) — {len(patched)} patched '
        f'cells recorded while cell {driven} was driven with the injected-current train in '
        f'panel A. Postsynaptic cells show their mean membrane-potential response; a genuine '
        f'monosynaptic connection appears as a sustained depolarisation.*'))


#### 3.2.4 Channel usage and paradigm composition
How the collection is composed across the multineuron array: sweeps recorded per channel index, and the mix of experimental paradigms across the 47 sessions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

chan = acquisition_series[acquisition_series['channel_num'] >= 0]['channel_num'].value_counts().sort_index()
axes[0].bar(chan.index.astype(int), chan.values, color='#55A868', edgecolor='white')
axes[0].set_xlabel('Recording channel (cell index)'); axes[0].set_ylabel('Sweeps')
axes[0].set_title('Sweeps per recording channel')

pc = recording_summary['paradigm'].value_counts()
axes[1].bar(pc.index, pc.values, color='#4C72B0', edgecolor='white')
axes[1].set_ylabel('# sessions'); axes[1].set_title('Session composition by paradigm')
axes[1].tick_params(axis='x', rotation=35)
for x, v in enumerate(pc.values):
    axes[1].text(x, v + 0.3, str(v), ha='center', fontsize=8)

plt.tight_layout(); plt.show()

## 4. Conclusion
This notebook walked through a first exploration of the LNMC layer 5 pyramidal connectivity whole-cell patch-clamp dataset:

- **Loading**: Croissant metadata (`fair2.json`) drove the ingestion of the three tidy index record-sets — 47 per-session `session_metadata` rows, 150,121 per-sweep `acquisition_series` rows, and 482 per-cell `electrode_metadata` rows.
- **Data dictionary with method provenance**: every column in every table is linked to a specific section in `methods_description` via `method_description_object`.
- **Recording composition**: up to twelve cells patched simultaneously (median 12), with a right-skewed distribution of sweeps per session.
- **Stimulus repertoire**: the distribution of sweeps across the 160 distinct Igor-Pro protocols.
- **Electrophysiology viewer**: simultaneous multineuron traces read on demand from the canonical NWB files, and a spike-triggered average revealing a genuine monosynaptic EPSP (cell 5 → cell 7).
- **Channel usage and paradigm composition**: sweeps per channel across the array and the mix of experimental paradigms.

Everything shown here is reproducible from the FAIR² package alone: the Croissant `recordSet` entries define the tabular index (with method-step linkage), and the raw signal for every sweep is preserved in the canonical NWB files (archived in `nwb_ephys_files.zip`). To extend, consider (a) a rigorous, latency-gated connectivity screen across all directed pairs (controlling for inter-electrode crosstalk), (b) extracting quantitative synaptic features (EPSP amplitude, latency, failure rate), or (c) relating connection probability to intersomatic distance — the central result of Perin et al. (2011).